# SupportGen · 04 — Evaluating the fine-tuned model (Model A vs Model B)
We compare **base FLAN-T5-small (A)** with **our fine-tuned FLAN-T5-small (B)** on the *same* fixed
test tickets, using:

1. Automatic metrics — ROUGE-1/2/L, BLEU, BERTScore, semantic cosine similarity, distinct-n
2. **Statistical significance** of the difference (paired Wilcoxon test + bootstrap confidence interval)
3. Efficiency — parameters, size on disk, latency (GPU and CPU), peak GPU memory
4. A blinded **human-evaluation** sheet (Relevance, Correctness, Fluency, Professional tone, Helpfulness)
5. Zero-shot **summary** and **recommended action** generation (no ground-truth labels exist for these)

**Runtime:** GPU. Requires notebooks 01 and 03 (02 is reused if it was run).

## 0. Setup — Google Drive & project folder
Every SupportGen notebook starts with this same cell. It mounts Google Drive so that all
notebooks share one project folder (`MyDrive/SupportGen`): notebook 01 writes the cleaned
splits there, notebooks 02–06 read them. Nothing ever needs to be uploaded from your laptop.

Set `USE_DRIVE = False` only for a throw-away run — files then live in `/content` and are lost
when the Colab runtime disconnects.

In [ ]:
# ── Setup (identical in every SupportGen notebook) ──────────────────────────
USE_DRIVE = True   # False → keep everything in /content (deleted when the runtime ends)

import os, sys
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/SupportGen"
else:
    BASE = "/content/SupportGen"
os.makedirs(BASE, exist_ok=True)
os.environ["SUPPORTGEN_BASE"] = BASE   # read by supportgen_common.py
sys.path.insert(0, BASE)
print("Project folder:", BASE)

In [ ]:
# Colab already ships torch, transformers, accelerate, scikit-learn, pandas, matplotlib.
# These are the extras SupportGen needs (already-installed packages are skipped).
!pip -q install evaluate rouge_score bert_score sacrebleu sentencepiece sentence-transformers

In [ ]:
# Shared helpers were written to Drive by 01_dataset_preparation.ipynb
if not os.path.exists(os.path.join(BASE, "supportgen_common.py")):
    raise FileNotFoundError(
        f"supportgen_common.py not found in {BASE}. Run 01_dataset_preparation.ipynb first "
        "(with the same USE_DRIVE setting).")
import importlib, supportgen_common as sg
importlib.reload(sg)
sg.set_seed()
print("Device:", sg.get_device())

In [ ]:
import json, time, textwrap
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

FT_RUN, FT_DIR = sg.RUN_FT, sg.FINETUNED_DIR        # change to sg.RUN_FT_BASE / sg.FINETUNED_BASE_DIR for Model D
BASE_RUN = sg.RUN_BASE
if not os.path.exists(os.path.join(FT_DIR, "training_config.json")):
    raise FileNotFoundError(f"No fine-tuned model in {FT_DIR} — run 03_train_flan_t5.ipynb first.")
train_cfg = json.load(open(os.path.join(FT_DIR, "training_config.json")))
MAX_IN, MAX_OUT = train_cfg["max_input_len"], train_cfg["max_target_len"]
eval_df = sg.load_eval_subset()
prompts = [sg.build_prompt(s, b) for s, b in zip(eval_df.subject, eval_df.body)]
device = sg.get_device()
print(f"{len(eval_df)} evaluation tickets | fine-tuned model: {FT_DIR}")

## 1. Generate with both models
Base-model outputs from notebook 02 are reused if present (same tickets, same decoding settings);
otherwise they are generated here.

In [ ]:
def run_model(model_dir_or_name, run_name):
    tok = AutoTokenizer.from_pretrained(model_dir_or_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_dir_or_name).to(device)
    preds, lat = sg.generate_batch(model, tok, prompts, MAX_IN, MAX_OUT, batch_size=16)
    out = eval_df[["uid", "subject", "body", "answer", "queue", "priority", "type"]].rename(columns={"answer": "reference"})
    out["prediction"], out["latency_s_batched"] = preds, lat
    out.to_csv(sg.outputs_path(run_name), index=False)
    return out, model, tok

if os.path.exists(sg.outputs_path(BASE_RUN)):
    base_out = pd.read_csv(sg.outputs_path(BASE_RUN), keep_default_na=False)
    assert base_out.uid.tolist() == eval_df.uid.tolist(), "baseline outputs were made on a different subset"
    print("Re-using base outputs from notebook 02")
else:
    base_out, _, _ = run_model(sg.BASE_MODEL, BASE_RUN)
ft_out, ft_model, ft_tok = run_model(FT_DIR, FT_RUN)
print("done")

## 2. Automatic metrics

In [ ]:
m_base, per_base = sg.compute_gen_metrics(base_out.prediction.tolist(), base_out.reference.tolist(), return_per_example=True)
m_ft, per_ft = sg.compute_gen_metrics(ft_out.prediction.tolist(), ft_out.reference.tolist(), return_per_example=True)
table = pd.DataFrame({"A: base FLAN-T5-small": m_base, "B: fine-tuned": m_ft})
display(table)

## 3. Is the difference statistically significant?
Both models answered the **same** tickets, so we use **paired** tests on per-ticket scores:
* **Wilcoxon signed-rank test** — non-parametric; p < 0.05 means the per-ticket differences are unlikely
  to be due to chance.
* **Bootstrap 95 % confidence interval** of the mean difference (B − A), 2,000 resamples.

In [ ]:
from scipy.stats import wilcoxon
rng = np.random.default_rng(sg.SEED)
sig = {}
for col in ["rouge1", "rouge2", "rougeL", "bertscore_f1", "semantic_cosine"]:
    diff = per_ft[col].values - per_base[col].values
    boot = [rng.choice(diff, len(diff)).mean() for _ in range(2000)]
    p = wilcoxon(per_ft[col], per_base[col]).pvalue if np.any(diff != 0) else 1.0
    sig[col] = {"mean_A": per_base[col].mean(), "mean_B": per_ft[col].mean(), "mean_diff_B_minus_A": diff.mean(),
                "ci95_low": np.percentile(boot, 2.5), "ci95_high": np.percentile(boot, 97.5),
                "wilcoxon_p": p, "B_better_on_%_of_tickets": float((diff > 0).mean() * 100)}
sig_df = pd.DataFrame(sig).T
display(sig_df.round(4))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["rougeL", "bertscore_f1_rescaled", "semantic_cosine"]):
    ax.boxplot([per_base[col], per_ft[col]]); ax.set_xticks([1, 2], ["A: base", "B: fine-tuned"]); ax.set_title(col)
plt.tight_layout(); plt.savefig(os.path.join(sg.PLOTS_DIR, "metric_boxplots_A_vs_B.png"), dpi=130, bbox_inches="tight"); plt.show()

## 4. Efficiency: size, memory and latency
Latency is measured with **batch size 1** (one ticket at a time, like the web app), both on the
Colab GPU and on CPU (closer to running the model on a laptop without a GPU).

In [ ]:
def dir_size_mb(path):
    return sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(path) for f in fs) / 1e6

def latency(model, tok, n, dev):
    model = model.to(dev)
    sub = prompts[:n]
    _ = sg.generate_batch(model, tok, sub[:2], MAX_IN, MAX_OUT, batch_size=1)    # warm-up
    if dev == "cuda": torch.cuda.reset_peak_memory_stats()
    _, lat = sg.generate_batch(model, tok, sub, MAX_IN, MAX_OUT, batch_size=1)
    peak = torch.cuda.max_memory_allocated() / 1e6 if dev == "cuda" else None
    return {"mean_s": float(np.mean(lat)), "p50_s": float(np.median(lat)), "p95_s": float(np.percentile(lat, 95)), "n": n, "peak_gpu_mb": peak}

eff = {"parameters": sum(p.numel() for p in ft_model.parameters()),
       "size_on_disk_mb": dir_size_mb(FT_DIR),
       "fp32_weights_mb_estimate": sum(p.numel() for p in ft_model.parameters()) * 4 / 1e6}
if device == "cuda":
    eff["latency_gpu_batch1"] = latency(ft_model, ft_tok, 30, "cuda")
eff["latency_cpu_batch1"] = latency(ft_model, ft_tok, 10, "cpu")
eff["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
ft_model.to(device)
display(pd.json_normalize(eff).T)

## 5. Qualitative comparison
Automatic metrics compare against one reference answer; reading examples shows *how* the outputs differ.

In [ ]:
from IPython.display import HTML
ex = pd.DataFrame({"ticket": "<b>" + eval_df.subject + "</b><br>" + eval_df.body.str.slice(0, 500),
                   "reference": eval_df.answer, "A: base": base_out.prediction, "B: fine-tuned": ft_out.prediction}).sample(8, random_state=sg.SEED)
ex.to_csv(os.path.join(sg.RESULTS_DIR, "qualitative_examples_A_vs_B.csv"), index=False)
HTML(ex.to_html(escape=False, index=False).replace("<td>", '<td style="text-align:left;vertical-align:top;max-width:380px">'))

## 6. Human evaluation sheet (blinded)
Automatic metrics cannot judge correctness or tone reliably, so we rate a sample by hand.

**Protocol**
1. The cell below writes `results/human_eval_sheet.csv` — 50 tickets × 2 responses. For each ticket the
   two responses are labelled **X** and **Y** in random order, so raters do not know which model wrote which.
2. Download it (or open it from Drive in Google Sheets), and give each response a score of **1–5** for
   `relevance, correctness, fluency, professional_tone, helpfulness`.
   Ideally 2–3 team members rate independently (save as `human_eval_filled_<name>.csv`).
3. Upload the filled file(s) back into `SupportGen/results/` and run the aggregation cell.

The secret key (`human_eval_key.csv`) maps X/Y back to models — don't look at it while rating.

In [ ]:
N_HUMAN = 50
CRITERIA = ["relevance", "correctness", "fluency", "professional_tone", "helpfulness"]
hs = eval_df.sample(N_HUMAN, random_state=sg.SEED).index
rows, key = [], []
for i in hs:
    pair = [("A_base", base_out.prediction[i]), ("B_finetuned", ft_out.prediction[i])]
    rng.shuffle(pair)
    for label, (system, text) in zip(["X", "Y"], pair):
        rows.append({"item": int(i), "subject": eval_df.subject[i], "ticket": eval_df.body[i],
                     "response_label": label, "response": text, **{c: "" for c in CRITERIA}, "comment": ""})
        key.append({"item": int(i), "response_label": label, "system": system})
pd.DataFrame(rows).to_csv(os.path.join(sg.RESULTS_DIR, "human_eval_sheet.csv"), index=False)
pd.DataFrame(key).to_csv(os.path.join(sg.RESULTS_DIR, "human_eval_key.csv"), index=False)
print("Wrote human_eval_sheet.csv and human_eval_key.csv to", sg.RESULTS_DIR)

In [ ]:
# ── Aggregate filled human-evaluation sheets (re-run after uploading them) ──
import glob
filled = sorted(glob.glob(os.path.join(sg.RESULTS_DIR, "human_eval_filled*.csv")))
if not filled:
    print("No filled sheets yet — upload human_eval_filled_<name>.csv to", sg.RESULTS_DIR)
    human = None
else:
    key_df = pd.read_csv(os.path.join(sg.RESULTS_DIR, "human_eval_key.csv"))
    allr = pd.concat([pd.read_csv(f).assign(rater=os.path.basename(f)) for f in filled])
    allr = allr.merge(key_df, on=["item", "response_label"])
    allr[CRITERIA] = allr[CRITERIA].apply(pd.to_numeric, errors="coerce")
    human = allr.groupby("system")[CRITERIA].agg(["mean", "std", "count"]).round(2)
    display(human)
    sg.save_json({s: allr[allr.system == s][CRITERIA].mean().to_dict() for s in allr.system.unique()} |
                 {"raters": len(filled), "items": int(allr["item"].nunique())},
                 os.path.join(sg.RESULTS_DIR, "human_eval_summary.json"))

## 7. Ticket summarisation and recommended action (zero-shot)
The dataset has **no reference summaries or reference actions**, so we do not train on, or compute
ROUGE for, these tasks — that would require inventing labels. Instead we use the instruction-following
ability FLAN-T5 already has (*zero-shot prompting*) and judge quality qualitatively / by human rating.

We use the **original** FLAN-T5 (not our fine-tuned model) for these tasks: fine-tuning specialised our
model to write replies, so it no longer follows other instructions as well. `flan-t5-base` is used here
because summarisation quality is noticeably weaker with the small model; change `AUX_MODEL` to compare.

In [ ]:
AUX_MODEL = "google/flan-t5-base"
aux_tok = AutoTokenizer.from_pretrained(AUX_MODEL)
aux = AutoModelForSeq2SeqLM.from_pretrained(AUX_MODEL).to(device)
sub = eval_df.sample(15, random_state=1)
summ, _ = sg.generate_batch(aux, aux_tok, [sg.build_prompt(s, b, sg.SUMMARY_TEMPLATE) for s, b in zip(sub.subject, sub.body)], MAX_IN, 64)
act, _ = sg.generate_batch(aux, aux_tok, [sg.build_prompt(s, b, sg.ACTION_TEMPLATE) for s, b in zip(sub.subject, sub.body)], MAX_IN, 48)
aux_df = pd.DataFrame({"subject": sub.subject.values, "ticket": sub.body.values, "generated_summary": summ, "recommended_action": act,
                       "summary_faithfulness_1to5": "", "summary_conciseness_1to5": "", "action_appropriateness_1to5": ""})
aux_df.to_csv(os.path.join(sg.RESULTS_DIR, "summary_action_examples.csv"), index=False)
for _, r in aux_df.head(6).iterrows():
    print("=" * 100); print("TICKET:", textwrap.shorten(r.subject + " — " + r.ticket, 350))
    print("SUMMARY:", r.generated_summary); print("ACTION :", r.recommended_action)

## 8. Save all results

In [ ]:
gen_metrics = {
    "eval_subset_size": len(eval_df),
    "models": {BASE_RUN: m_base, FT_RUN: m_ft},
    "significance_B_vs_A": sig,
    "efficiency_finetuned": eff,
    "fine_tuned_model_dir": FT_DIR, "training_config": train_cfg,
    "aux_model_for_summary_action": AUX_MODEL,
    "human_eval": None if human is None else sg.load_json(os.path.join(sg.RESULTS_DIR, "human_eval_summary.json")),
    "notes": "ROUGE/BERTScore/cosine on 0-1 scale; BLEU on 0-100. All models use the same prompt and decoding settings.",
}
sg.save_json(gen_metrics, os.path.join(sg.RESULTS_DIR, "gen_metrics.json"))
sg.save_json(m_ft | {"run_name": FT_RUN}, sg.metrics_path(FT_RUN))
for run, m in [(BASE_RUN, m_base), (FT_RUN, m_ft)]:
    sg.log_experiment(run, rouge1=m["rouge1"], rouge2=m["rouge2"], rougeL=m["rougeL"], bleu=m["bleu"],
                      bertscore_f1=m["bertscore_f1"], semantic_cosine=m["semantic_cosine"])
display(pd.read_csv(sg.EXPERIMENTS_CSV))

## How to state the conclusion honestly
* Claim "fine-tuning improved domain-specific response generation" **only if** B beats A on the metrics
  *and* the Wilcoxon p-values are < 0.05 / the bootstrap CIs exclude 0 — and ideally the human ratings agree.
* If some metrics improve and others don't, report exactly that (e.g. higher ROUGE but similar BERTScore).
* Remember the dataset is synthetic: results show adaptation to this dataset's style and domain.

➡️ **Next:** `05_classification_models.ipynb` (CPU runtime is enough).